# Export POC-ABS dari apex event clips ke satu XLSX

Pipeline:
1. Baca `output/apex/metadata.xlsx`
2. Ambil semua clip event dari `output/apex/dataset/.../event_*`
3. Per event clip: frame pertama = baseline, frame berikutnya = POC-ABS flatten
4. Gabung semua row jadi **satu XLSX**
5. Urutkan per partisipan / label / pertanyaan dari **before → after**


In [1]:
import os
import re
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'preprocess-anxiety':
    ROOT = ROOT.parent.resolve()

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')


Project root: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st


In [2]:
from comparasion.core.config import ComparisonConfig
from comparasion.core.roi import ROIExtractor
from features_extraction.poc import POC
from features_extraction.quadran import Quadran
from features_extraction.vektor import Vektor


In [3]:
DATASET_ROOT = (ROOT / 'output/apex/dataset').resolve()
METADATA_PATH = (ROOT / 'output/apex/metadata.xlsx').resolve()
OUTPUT_DIR = (ROOT / 'output/apex/features').resolve()
OUTPUT_PATH = OUTPUT_DIR / 'poc_abs_flatten_ordered.xlsx'
MAX_WORKERS = min(8, os.cpu_count() or 1)

REGIONS = {
    'mulut': list(range(48, 68)),
    'mata_kiri': list(range(17, 22)) + list(range(36, 42)),
    'mata_kanan': list(range(22, 27)) + list(range(42, 48)),
}

TARGET_SIZE = {
    'mulut': (70, 35),
    'mata_kiri': (48, 32),
    'mata_kanan': (48, 32),
}

config = ComparisonConfig(
    output_root=Path('output/apex/tmp_compare'),
    regions=REGIONS,
    target_size=TARGET_SIZE,
)

print(f'Dataset root: {DATASET_ROOT}')
print(f'Metadata:     {METADATA_PATH}')
print(f'Output xlsx:  {OUTPUT_PATH}')
print(f'Max workers:  {MAX_WORKERS}')
print(f'Regions:      {list(config.regions.keys())}')
print(f'Target size:  {config.target_size}')
print(f'Block size:   {config.block_size}')


Dataset root: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/dataset
Metadata:     /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/metadata.xlsx
Output xlsx:  /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/features/poc_abs_flatten_ordered.xlsx
Max workers:  8
Regions:      ['mulut', 'mata_kiri', 'mata_kanan']
Target size:  {'mulut': (70, 35), 'mata_kiri': (48, 32), 'mata_kanan': (48, 32)}
Block size:   7


In [4]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset root not found: {DATASET_ROOT}')
if not METADATA_PATH.exists():
    raise FileNotFoundError(f'Metadata not found: {METADATA_PATH}')
if not config.predictor_path.exists():
    raise FileNotFoundError(f'Predictor not found: {config.predictor_path}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata = pd.read_excel(METADATA_PATH)
print(f'Metadata rows: {len(metadata)}')
metadata.head(2)


Metadata rows: 2764


,relative_sample,source_video,event_no,onset_signal,apex_signal,offset_signal,onset_frame,apex_frame,offset_frame,duration,saved_frames,window_length,polyorder,height_threshold,clip_dir
0,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,1,93,98,103,94,99,104,10,11,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...
1,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,2,175,205,212,176,206,213,37,38,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...


In [5]:
@dataclass(slots=True)
class EventClip:
    phase: str
    condition: str
    label: str
    participant_raw: str
    participant_key: str
    question: str
    question_no: int
    sample_name: str
    clip_name: str
    event_clip: str
    event_no: int
    event_dir: Path
    onset_frame: int
    apex_frame: int
    offset_frame: int

    @property
    def sample_id(self) -> str:
        return f'{self.phase}__{self.condition}__{self.participant_raw}__{self.question}__{self.sample_name}__event_{self.event_no:03d}'


def natural_sort_key(value: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', value)]


def parse_question_no(question: str) -> int:
    match = re.search(r'(\d+)', question)
    return int(match.group(1)) if match else 0


def normalize_participant_key(name: str) -> str:
    return re.sub(r'_[0-9]+$', '', name)


def map_label(condition: str) -> str:
    mapping = {
        'anxiety': 'anxiety_tinggi',
        'tidak': 'anxiety_rendah',
    }
    return mapping.get(condition, condition)


def parse_event_dir(clip_dir_value: str) -> Path:
    raw = Path(str(clip_dir_value).replace('\\', '/'))
    parts = raw.parts

    if raw.is_absolute():
        return raw.resolve()

    if parts[:3] == ('output', 'apex', 'dataset'):
        raw = Path(*parts[3:])
    elif parts and parts[0] == 'dataset':
        raw = Path(*parts[1:])

    return (DATASET_ROOT / raw).resolve()


def build_event_clips(df: pd.DataFrame) -> list[EventClip]:
    clips: list[EventClip] = []
    valid = df[df['clip_dir'].notna()].copy()

    for row in valid.itertuples(index=False):
        parts = Path(row.relative_sample).parts
        if len(parts) < 5:
            continue

        phase, condition, participant_raw, question, sample_name = parts[:5]
        event_dir = parse_event_dir(row.clip_dir)
        clips.append(EventClip(
            phase=phase,
            condition=condition,
            label=map_label(condition),
            participant_raw=participant_raw,
            participant_key=normalize_participant_key(participant_raw),
            question=question,
            question_no=parse_question_no(question),
            sample_name=sample_name,
            clip_name=sample_name,
            event_clip=event_dir.name,
            event_no=int(row.event_no),
            event_dir=event_dir,
            onset_frame=int(row.onset_frame),
            apex_frame=int(row.apex_frame),
            offset_frame=int(row.offset_frame),
        ))

    return sorted(
        clips,
        key=lambda c: (
            c.participant_key,
            c.label,
            c.question_no,
            0 if c.phase == 'before' else 1,
            natural_sort_key(c.clip_name),
            c.event_no,
            natural_sort_key(c.event_clip),
        ),
    )


event_clips = build_event_clips(metadata)
print(f'Event clips: {len(event_clips)}')
event_clips[:3]


Event clips: 2764


[EventClip(phase='before', condition='anxiety', label='anxiety_tinggi', participant_raw='aaisyah_nursalsabiil_ni_patriarti_1765168488512', participant_key='aaisyah_nursalsabiil_ni_patriarti', question='q1', question_no=1, sample_name='answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec', clip_name='answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec', event_clip='event_00054-00095', event_no=1, event_dir=PosixPath('/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/dataset/before/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765168488512/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00054-00095'), onset_frame=54, apex_frame=65, offset_frame=95),
 EventClip(phase='before', condition='anxiety', label='anxiety_tinggi', participant_raw='aaisyah_nursalsabiil_ni_patriarti_1765168488512', participant_key='aaisyah_nursalsabiil_ni_patriarti', question='q1', question_no=1, sample_name='answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec', clip_name='answer_1_15d591ce-051a-47f2

In [6]:
WORKER_EXTRACTOR = None
WORKER_CONFIG = None


def init_worker() -> None:
    global WORKER_EXTRACTOR, WORKER_CONFIG
    WORKER_CONFIG = ComparisonConfig(
        output_root=Path('output/apex/tmp_compare'),
        regions=REGIONS,
        target_size=TARGET_SIZE,
        padding_x=config.padding_x,
        padding_y=config.padding_y,
        block_size=config.block_size,
        predictor_path=config.predictor_path,
        feature_method=config.feature_method,
        include_quadrant_counts=config.include_quadrant_counts,
    )
    WORKER_EXTRACTOR = ROIExtractor(WORKER_CONFIG)


def load_event_frames(event_dir: Path) -> tuple[list[np.ndarray], list[Path]]:
    frame_paths = sorted(
        [p for p in event_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}],
        key=lambda p: natural_sort_key(p.name),
    )
    if len(frame_paths) < 2:
        raise ValueError(f'Need >= 2 frames: {event_dir}')

    frames = []
    for path in frame_paths:
        image = cv2.imread(str(path))
        if image is None:
            raise FileNotFoundError(f'Failed to read image: {path}')
        frames.append(image)
    return frames, frame_paths


def to_gray(image: np.ndarray) -> np.ndarray:
    if image.ndim == 2:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


def extract_flatten_row(clip: EventClip, frame_no: int, rois: dict[str, np.ndarray], baseline_gray: dict[str, np.ndarray], block_size: int) -> dict:
    row = {
        'phase': clip.phase,
        'condition': clip.condition,
        'label': clip.label,
        'participant': clip.participant_key,
        'participant_raw': clip.participant_raw,
        'question': clip.question,
        'question_no': clip.question_no,
        'sample': clip.sample_name,
        'clip': clip.clip_name,
        'event_clip': clip.event_clip,
        'event_no': clip.event_no,
        'clip_path': str(clip.event_dir.resolve().relative_to(ROOT.resolve())),
        'frame': frame_no,
    }

    for comp in WORKER_CONFIG.regions:
        gray = to_gray(rois[comp])
        poc = POC(baseline_gray[comp], gray, block_size)
        vec = Vektor(poc.getPOC(), block_size)
        quad = Quadran(vec.getVektor()).getQuadran()

        for block_id, block in enumerate(quad, start=1):
            row[f'{comp}_x{block_id}'] = block[1]
            row[f'{comp}_y{block_id}'] = block[2]
            row[f'{comp}_t{block_id}'] = block[3]
            row[f'{comp}_m{block_id}'] = block[4]

    return row


def process_event_clip(clip: EventClip) -> list[dict]:
    frames, _frame_paths = load_event_frames(clip.event_dir)

    rois_per_frame: list[dict[str, np.ndarray] | None] = []
    for frame in frames:
        try:
            rois_per_frame.append(WORKER_EXTRACTOR.extract_rois(frame))
        except ValueError:
            rois_per_frame.append(None)

    baseline_rois = rois_per_frame[0]
    if baseline_rois is None:
        raise ValueError(f'No face detected in baseline frame: {clip.event_dir}')

    baseline_gray = {name: to_gray(img) for name, img in baseline_rois.items()}
    rows: list[dict] = []

    for idx in range(1, len(frames)):
        rois = rois_per_frame[idx]
        if rois is None:
            continue

        frame_no = idx + 1
        rows.append(extract_flatten_row(clip, frame_no, rois, baseline_gray, WORKER_CONFIG.block_size))

    return rows


def worker_task(clip: EventClip) -> tuple[EventClip, list[dict] | None, str | None]:
    try:
        return clip, process_event_clip(clip), None
    except Exception as exc:
        return clip, None, str(exc)


In [7]:
all_rows = []
errors = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=init_worker) as executor:
    futures = [executor.submit(worker_task, clip) for clip in event_clips]

    for idx, future in enumerate(futures, start=1):
        clip, rows, error = future.result()
        print(f'[{idx}/{len(event_clips)}] {clip.phase}/{clip.label}/{clip.participant_key}/{clip.question}/{clip.clip_name}/{clip.event_clip}/event_{clip.event_no:03d}')
        if error is not None:
            errors.append((clip.sample_id, error))
            continue
        all_rows.extend(rows)

print(f'Rows collected: {len(all_rows)}')
print(f'Errors: {len(errors)}')


[1/2764] before/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00054-00095/event_001
[2/2764] before/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00153-00166/event_002
[3/2764] before/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00173-00187/event_003
[4/2764] before/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00253-00272/event_004
[5/2764] before/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00284-00313/event_005
[6/2764] after/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00160-00186/event_001
[7/2764] after/anxiety_tinggi/aaisyah_nursalsabiil_ni_patriarti/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00

In [8]:
if errors:
    error_df = pd.DataFrame(errors, columns=['sample_id', 'error'])
    print(error_df.head(20).to_string(index=False))
    error_df.head(20)


                                                                                                            sample_id                                                                                                                                                                                                                                  error
before__tidak__ashrul_rifki_ardiyhasa_1765264624251__q1__answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec__event_002 No face detected in baseline frame: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/dataset/before/tidak/ashrul_rifki_ardiyhasa_1765264624251/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec/event_00102-00124


In [9]:
df = pd.DataFrame(all_rows)
if df.empty:
    raise ValueError('No rows produced.')

df = df.sort_values(
    by=['participant', 'label', 'question_no', 'phase', 'clip', 'event_clip', 'event_no', 'frame'],
    key=lambda s: s.map({'before': 0, 'after': 1}) if s.name == 'phase' else s,
    kind='stable',
).reset_index(drop=True)

df.to_excel(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df.shape}')
df.head()


Saved: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/features/poc_abs_flatten_ordered.xlsx
Shape: (57656, 405)


,phase,condition,label,participant,participant_raw,question,question_no,sample,clip,event_clip,...,mata_kanan_t22,mata_kanan_m22,mata_kanan_x23,mata_kanan_y23,mata_kanan_t23,mata_kanan_m23,mata_kanan_x24,mata_kanan_y24,mata_kanan_t24,mata_kanan_m24
0,before,anxiety,anxiety_tinggi,aaisyah_nursalsabiil_ni_patriarti,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q1,1,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,event_00054-00095,...,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0
1,before,anxiety,anxiety_tinggi,aaisyah_nursalsabiil_ni_patriarti,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q1,1,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,event_00054-00095,...,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0
2,before,anxiety,anxiety_tinggi,aaisyah_nursalsabiil_ni_patriarti,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q1,1,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,event_00054-00095,...,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0
3,before,anxiety,anxiety_tinggi,aaisyah_nursalsabiil_ni_patriarti,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q1,1,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,event_00054-00095,...,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0
4,before,anxiety,anxiety_tinggi,aaisyah_nursalsabiil_ni_patriarti,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q1,1,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec,event_00054-00095,...,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0


In [11]:
PHASE_SORT = {'before': 0, 'after': 1}

df_phase_first = df.assign(
    _phase_order=df['phase'].map(PHASE_SORT).fillna(99),
).sort_values(
    by=['_phase_order', 'participant', 'label', 'question_no', 'clip', 'event_clip', 'event_no', 'frame'],
    kind='stable',
).drop(columns=['_phase_order']).reset_index(drop=True)

df_phase_first.to_excel(OUTPUT_PATH, index=False)
df = df_phase_first

print(f'Re-saved with phase-first ordering: {OUTPUT_PATH}')
print(df[['phase', 'participant', 'label', 'question', 'clip', 'event_clip', 'event_no', 'frame']].head(20).to_string(index=False))


Re-saved with phase-first ordering: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/features/poc_abs_flatten_ordered.xlsx
 phase                       participant          label question                                              clip        event_clip  event_no  frame
before aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      2
before aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      3
before aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      4
before aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      5
before aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7

In [10]:
print(df[['participant', 'label', 'question', 'phase', 'clip', 'event_clip', 'event_no', 'frame']].head(20).to_string(index=False))
print()
print(df['phase'].value_counts().to_dict())
print(df['label'].value_counts().to_dict())


                      participant          label question  phase                                              clip        event_clip  event_no  frame
aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 before answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      2
aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 before answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      3
aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 before answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      4
aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 before answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      5
aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 before answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec event_00054-00095         1      6
aaisyah_nursalsabiil_ni_patriarti anxiety_tinggi       q1 before answer_1_15d591ce-051a-47f2-ac38-36